# Láminas para colorear con SDXL (gratis en Colab)

**Solo dos pasos:** pulsa **Conectar** (arriba a la derecha) y luego
*Entorno de ejecución → Ejecutar todo*. Acepta el permiso de Google Drive cuando lo pida.

- Ya viene con GPU T4. Si dijera que no hay GPU: *Entorno de ejecución → Cambiar tipo → T4 GPU*.
- Tarda unas 2–3 horas. Guarda cada imagen en `Mi unidad/colorear_lotes/<categoría>/` al momento.
- Si Colab se desconecta, vuelve a *Ejecutar todo*: salta las que ya hizo y sigue.
- Al terminar crea `Mi unidad/colorear_zip/` con un .zip por categoría. Avisa a Claude:
  las lee directamente de tu Drive, no hace falta compartir nada.

Modelos: SDXL base 1.0 (Stability AI) + SDXL-Lightning 8 pasos (ByteDance), ambos con
licencia `openrail++`, que permite uso comercial respetando sus restricciones de uso.

In [ ]:
# Ajustes
MODO = "rapido"      # "rapido" (Lightning, ~5 s por imagen) o "calidad" (SDXL normal, ~25 s, usa el prompt negativo)
SEMILLAS = 2         # imágenes por prompt (855 prompts x 2 = 1710)
SOLO_PRUEBA = 0      # 0 = todas; un número (p. ej. 20) para hacer solo una prueba
CARPETA = "/content/drive/MyDrive/colorear_lotes"
CARPETA_ZIP = "/content/drive/MyDrive/colorear_zip"

In [ ]:
!pip -q install diffusers==0.31.0 transformers accelerate safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""Lista de prompts para generar las láminas en Colab (la usa generar_laminas.ipynb).

Cada entrada: (id, categoria, prompt). Se generan SEMILLAS imágenes por prompt.
Los estilos son libres a propósito: cada lámina puede ser única.
"""

FINAL = (", black and white coloring book page for adults, clean bold black outlines, pure white background, "
         "line art only, no shading, no gray, no color, no gradients, closed shapes, centered composition, highly detailed")

ESTILOS = {
    "tatuaje": "{s}, tattoo design sketch, elegant fine linework, flowing ornamental swirls",
    "etnico": "{s}, ethnic zentangle style, the body filled with tribal ornaments and patterns, symmetrical, intricate",
    "realista": "{s}, realistic detailed illustration with a background of mountains and pine trees",
    "mandala": "{s} merged with mandala patterns, intricate ornamental details",
    "geometrico": "{s}, geometric tattoo, sacred geometry lines and circles around it",
    "nouveau": "{s}, art nouveau style, decorative frame with flowers and curves",
    "botanico": "{s}, vintage botanical illustration, detailed engraving style lines",
    "grabado": "{s}, vintage engraving style illustration, detailed",
    "steampunk": "{s}, steampunk style with gears and ornate details",
    "paisaje": "{s}, detailed landscape illustration",
}

ANIMALES = ["lion head", "tiger head", "wolf head", "fox", "bear", "panda", "koala", "monkey", "gorilla", "elephant",
            "giraffe", "zebra", "horse head", "deer with antlers", "ram with curled horns", "bull head", "rhinoceros",
            "hippopotamus", "cheetah", "leopard", "jaguar", "lynx", "cat", "sleeping cat", "dog", "german shepherd", "husky",
            "rabbit", "hare", "squirrel", "raccoon", "hedgehog", "otter", "red panda", "sloth", "kangaroo", "camel",
            "llama", "alpaca", "bison", "moose", "boar", "goat", "sheep", "cow", "pig", "chameleon", "iguana", "crocodile",
            "snake coiled", "cobra", "turtle", "frog", "gecko", "bat", "mouse", "badger", "armadillo", "anteater", "capybara"]
AVES = ["eagle head", "owl", "barn owl", "hummingbird", "peacock", "parrot", "toucan", "flamingo", "swan", "crane bird",
        "raven", "hawk", "falcon", "rooster", "hen with chicks", "penguin", "pelican", "kingfisher", "phoenix-like bird of paradise", "swallow birds"]
OCEANO = ["koi fish", "whale", "humpback whale", "dolphin", "octopus", "jellyfish", "sea turtle", "seahorse", "shark", "manta ray",
          "starfish and shells", "crab", "lobster", "clownfish in coral", "pufferfish", "angelfish", "orca", "narwhal", "nautilus shell", "coral reef"]
INSECTOS = ["butterfly", "moth", "dragonfly", "beetle", "honeybee", "ladybug", "praying mantis", "scorpion", "spider on web",
            "grasshopper", "snail", "caterpillar on leaf", "firefly", "stag beetle", "bumblebee on flower"]
FANTASIA = ["dragon", "chinese dragon", "phoenix", "unicorn", "pegasus", "griffin", "mermaid tail and shells", "kitsune nine tailed fox",
            "fairy house in a mushroom", "wizard castle", "treasure chest", "magic potion bottles", "crystal ball", "enchanted forest",
            "kraken", "hydra", "sea serpent", "centaur", "cerberus", "thunderbird", "fantasy owl with crown", "moon goddess cat",
            "floating island", "steampunk airship", "fairy wings", "dragon egg", "magic book", "fantasy sword", "witch hat and moon", "baby dragon"]
FLORES = ["rose", "lotus", "sunflower", "peony", "orchid", "lily", "tulips", "daisy bouquet", "cherry blossom branch", "hibiscus",
          "magnolia", "poppy", "lavender", "chrysanthemum", "dahlia", "iris flower", "camellia", "succulent plants", "cactus in pot", "fern leaves",
          "monstera leaves", "bonsai tree", "tree of life", "oak tree", "willow tree", "wreath of flowers", "flower vase", "mushrooms", "wildflowers", "water lily"]
COMIDA = ["apple", "pear", "strawberries", "cherries", "grapes", "orange slices", "lemon branch", "pineapple", "watermelon slice", "mango",
          "avocado", "pomegranate", "coconut", "banana bunch", "blueberries", "fig", "kiwi slices", "peach", "fruit basket", "cupcakes",
          "layered cake", "ice cream cones", "donuts", "coffee cup with steam", "teapot and cups", "macarons", "chocolate box", "pizza", "sushi set", "bread basket"]
OBJETOS = ["vintage pocket watch", "old key", "lantern", "teapot", "hot air balloon", "vintage camera", "gramophone", "typewriter", "compass",
           "hourglass", "antique clock", "violin", "guitar", "piano", "saxophone", "drum", "music box", "chandelier", "candle holder",
           "old books stack", "globe", "telescope", "anchor", "ship wheel", "dreamcatcher", "perfume bottles", "jewelry box", "crown", "masquerade mask",
           "fan with patterns", "umbrella", "bicycle", "sewing machine", "birdcage", "kettle", "rocking chair", "lamp", "teacup", "vase with patterns", "chess pieces"]
VEHICULOS = ["vintage car", "classic motorcycle", "steam locomotive", "sailing ship", "pirate ship", "submarine", "biplane", "rocket",
             "vintage truck", "tram", "hot rod car", "vespa scooter", "canoe on a lake", "lighthouse and boat", "gondola", "camper van",
             "fire truck", "tractor", "carriage", "zeppelin"]
LUGARES = ["cabin in the woods", "lighthouse on cliffs", "waterfall in forest", "mountain lake", "japanese temple", "european street with cafes",
           "medieval castle", "windmill in fields", "tropical beach with palm trees", "cottage with garden", "old town houses", "bridge over river",
           "greenhouse with plants", "treehouse", "village in valley", "desert with cactus", "snowy mountains", "rice terraces", "harbor with boats",
           "underwater city", "colonial street with balconies", "victorian house", "gothic cathedral", "pagoda", "farm with barn", "city skyline",
           "library interior", "cozy kitchen interior", "garden with fountain", "market stalls"]
MANDALAS = ["flower", "sun", "moon and stars", "ocean waves", "feathers", "leaves", "hearts", "butterflies", "lotus", "geometric shapes",
            "snowflakes", "fire", "peacock feathers", "seashells", "birds", "cats", "owls", "paisley", "celtic knots", "art deco",
            "fruits", "musical notes", "coffee", "cosmic planets", "roses", "mushrooms", "elephants", "fish", "dragons", "autumn"]

TANDAS = [
    ("animales", ANIMALES, ["tatuaje", "etnico", "realista", "mandala"]),
    ("aves", AVES, ["tatuaje", "etnico", "realista"]),
    ("oceano", OCEANO, ["tatuaje", "etnico", "mandala"]),
    ("insectos", INSECTOS, ["tatuaje", "etnico", "geometrico"]),
    ("fantasia", FANTASIA, ["tatuaje", "etnico", "nouveau"]),
    ("flores", FLORES, ["botanico", "mandala", "nouveau"]),
    ("comida", COMIDA, ["botanico", "etnico"]),
    ("objetos", OBJETOS, ["grabado", "steampunk"]),
    ("vehiculos", VEHICULOS, ["grabado", "paisaje"]),
    ("lugares", LUGARES, ["paisaje", "grabado"]),
]


def lista():
    salida = []
    for cat, temas, estilos in TANDAS:
        for i, tema in enumerate(temas):
            for e in estilos:
                salida.append(("%s_%03d_%s" % (cat, i + 1, e), cat, ESTILOS[e].format(s=tema) + FINAL))
    for i, tema in enumerate(MANDALAS):
        salida.append(("mandalas_%03d" % (i + 1), "mandalas", "intricate mandala made of " + tema + FINAL))
    return salida


if False:
    l = lista()
    print(len(l), "prompts")


In [ ]:
import os, torch
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler, AutoencoderKL
from huggingface_hub import hf_hub_download

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", vae=vae, torch_dtype=torch.float16, variant="fp16").to("cuda")
if MODO == "rapido":
    pipe.load_lora_weights(hf_hub_download("ByteDance/SDXL-Lightning", "sdxl_lightning_8step_lora.safetensors"))
    pipe.fuse_lora()
    pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")
    PASOS, GUIA = 8, 0.0
else:
    PASOS, GUIA = 30, 6.0
NEGATIVO = ("color, colored, gray, grey, shading, shadow, gradient, hatching, crosshatching, stippling, "
            "photo, 3d render, blurry, noisy, text, letters, watermark, signature, frame border, cropped")
print("Listo:", MODO, PASOS, "pasos")

In [ ]:
import time
trabajos = [(pid, cat, p, s) for pid, cat, p in lista() for s in range(SEMILLAS)]
if SOLO_PRUEBA:
    trabajos = trabajos[::max(1, len(trabajos) // SOLO_PRUEBA)][:SOLO_PRUEBA]
hechas, t0 = 0, time.time()
for n, (pid, cat, prompt, s) in enumerate(trabajos):
    os.makedirs(f"{CARPETA}/{cat}", exist_ok=True)
    ruta = f"{CARPETA}/{cat}/{pid}_s{s}.png"
    if os.path.exists(ruta):
        continue
    gen = torch.Generator("cuda").manual_seed(1000 + s * 7919 + n)
    img = pipe(prompt, negative_prompt=NEGATIVO if GUIA > 0 else None, num_inference_steps=PASOS,
               guidance_scale=GUIA, width=1024, height=1024, generator=gen).images[0]
    img.convert("L").save(ruta, optimize=True)
    hechas += 1
    if hechas % 10 == 0:
        ritmo = (time.time() - t0) / hechas
        print(f"{n + 1}/{len(trabajos)}  ({ritmo:.1f} s por imagen, faltan ~{ritmo * (len(trabajos) - n - 1) / 60:.0f} min)")
print("Terminado")

In [ ]:
# Un .zip por categoría para pasárselo a Claude
import shutil
os.makedirs(CARPETA_ZIP, exist_ok=True)
for cat in sorted(os.listdir(CARPETA)):
    shutil.make_archive(f"{CARPETA_ZIP}/{cat}", "zip", f"{CARPETA}/{cat}")
    print(cat, len(os.listdir(f"{CARPETA}/{cat}")), "imágenes")